# 02 — Preprocessing Pipeline

Walk through the full EEG preprocessing pipeline and save the resulting epoch arrays to disk for model training.

**Steps:**
1. Load raw data
2. Bandpass + notch filtering
3. Epoch segmentation
4. Artifact rejection
5. Normalization
6. Save to `data/processed/`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.loader import EEGDataLoader
from src.data.preprocessor import EEGPreprocessor
from src.utils.visualization import plot_eeg_traces, plot_psd

%matplotlib inline
TASK = 'sleep'   # change to 'focus' or 'stress'
print(f'Task: {TASK}')

## 1. Load Raw Data

In [ ]:
loader = EEGDataLoader(data_dir='../data/raw/sleep-edf')

all_epochs, all_labels = [], []
N_SUBJECTS = 5   # increase for a fuller dataset

for subj_id in range(N_SUBJECTS):
    try:
        subj = loader.load_sleep_edf(subject_id=subj_id, data_dir='../data/raw/sleep-edf')
    except FileNotFoundError:
        print(f'Subject {subj_id} not found — skipping')
        continue

    data   = subj['data']
    labels = subj['labels']
    sfreq  = subj['sfreq']

    # Keep only EEG channels (first 2 in Sleep-EDF)
    eeg_data = data[:2]

    # Expand sample-level labels from epoch-level (30s epochs)
    n_samples = eeg_data.shape[1]
    samples_per_epoch = int(30 * sfreq)
    sample_labels = np.repeat(labels[:n_samples // samples_per_epoch], samples_per_epoch)
    sample_labels = np.pad(sample_labels, (0, n_samples - len(sample_labels)), constant_values=-1)

    # Remove unlabelled samples
    valid_mask = sample_labels >= 0
    eeg_data = eeg_data[:, valid_mask]
    sample_labels = sample_labels[valid_mask]

    all_epochs.append((eeg_data, sample_labels, sfreq))
    print(f'Loaded subject {subj_id}: {eeg_data.shape}')

## 2. Preprocess & Epoch

In [ ]:
preprocessor = EEGPreprocessor(
    sfreq=sfreq,
    lowcut=0.3,
    highcut=35.0,
    notch_freq=50.0,
    epoch_length=30.0,   # 30-second AASM epochs for sleep staging
    overlap=0.0,
    amplitude_threshold=200.0,
)

epoch_list, label_list = [], []
for eeg_data, sample_labels, _ in all_epochs:
    X, y = preprocessor.fit_transform(eeg_data, sample_labels)
    epoch_list.append(X)
    label_list.append(y)

epochs = np.concatenate(epoch_list, axis=0)
labels = np.concatenate(label_list, axis=0)

print(f'\nFinal dataset: {epochs.shape}  |  labels: {labels.shape}')
print(f'Class distribution: {dict(zip(*np.unique(labels, return_counts=True)))}')

## 3. Visualise Before vs After Filtering

In [ ]:
raw_data = all_epochs[0][0]
filtered = preprocessor.bandpass_filter(preprocessor.notch_filter(raw_data))

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
t = np.arange(10 * sfreq) / sfreq
axes[0].plot(t, raw_data[0, :len(t)], lw=0.8, color='#e74c3c')
axes[0].set_title('Raw EEG — Channel 0', fontsize=12)
axes[1].plot(t, filtered[0, :len(t)], lw=0.8, color='#2ecc71')
axes[1].set_title('Filtered EEG (0.3–35 Hz, notch 50 Hz)', fontsize=12)
for ax in axes:
    ax.set_ylabel('Amplitude (µV)')
    ax.grid(alpha=0.3)
axes[1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## 4. Save Processed Data

In [ ]:
out_dir = Path('../data/processed')
out_dir.mkdir(parents=True, exist_ok=True)

np.save(out_dir / f'{TASK}_data.npy',   epochs)
np.save(out_dir / f'{TASK}_labels.npy', labels)

print(f'Saved {TASK}_data.npy   → {epochs.shape}')
print(f'Saved {TASK}_labels.npy → {labels.shape}')